# HCP Targeting Engine — synthetic interactive walkthrough

**Synthetic data only.** All analyses are exploratory and do not estimate causal impact or justify live promotional decisions.

In [ ]:
from hcp_engine import Config, run_pipeline
from hcp_engine.data import generate_synthetic
from hcp_engine_plus.audits import quality_report, group_score_audit
from hcp_engine_plus.dashboard import write_dashboard
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image, HTML

## Build a synthetic dataset and run the original pipeline

Set `k=5` and `n_hcps=800` so the demo completes reasonably quickly. Real SHAP runs, and results include scores and reason codes.

In [ ]:
data = generate_synthetic(n_hcps=800, seed=42)
quality_report(data, range(1,16))

In [ ]:
res = run_pipeline(Config(seed=42, k=5), data=data, out_dir="outputs/notebook", make_plots=True)
res.metrics["propensity_cv"]

## Targets and group score diagnostics

Score gaps by region are descriptive, not a protected-class fairness audit.

In [ ]:
display(res.target_table[["hcp_id", "segment", "status", "propensity", "expected_value", "why_target"]].head(12))
display(group_score_audit(res.target_table, "region", min_group_size=10))

## Real SHAP output on synthetic records

In [ ]:
display(Image(filename="outputs/notebook/figures/shap_importance.png"))
display(Image(filename="outputs/notebook/figures/shap_beeswarm.png"))

## Local searchable dashboard

In [ ]:
out = write_dashboard(res.target_table, "outputs/notebook/dashboard.html")
print(f"Open {out} in a browser to filter the target table.")

## What the notebook does not establish

These metrics reflect a single synthetic cohort. Temporal validation requires two genuine point-in-time snapshots with non-overlapping outcome windows and correctly cut off engagement data. See `docs/EXTENSIONS.md`.